In [0]:
# Configuration
VOLUME_PATH = "/Volumes/tws_ro_region5/rcd/pa_second_appeals"
CHUNKS_TABLE = "tws_ro_region5.rcd.pa_appeals_chunks_vs"
VS_INDEX = "tws_ro_region5.rcd.pa_appeals_chunks_vs_index"

In [0]:
from pyspark.sql import functions as F

# Recursively list all PDFs in the volume
def list_all_pdfs(base_path):
    """Recursively list all PDF files in the volume."""
    all_files = []
    dirs_to_scan = [base_path]
    while dirs_to_scan:
        current = dirs_to_scan.pop()
        try:
            entries = dbutils.fs.ls(current)
        except Exception:
            continue
        for entry in entries:
            if entry.name.endswith("/"):
                dirs_to_scan.append(entry.path)
            elif entry.name.lower().endswith(".pdf"):
                all_files.append({
                    "path": entry.path,
                    "filename": entry.name,
                    "size": entry.size,
                    "mod_time": entry.modificationTime
                })
    return all_files

volume_pdfs = list_all_pdfs(VOLUME_PATH)
print(f"Total PDFs in volume: {len(volume_pdfs)}")

Total PDFs in volume: 2871


In [0]:
# Get filenames already processed in the chunks table
existing_df = spark.sql(f"SELECT DISTINCT filename FROM {CHUNKS_TABLE}")
existing_filenames = set(row.filename for row in existing_df.collect())
print(f"Already processed: {len(existing_filenames)} files")

# Find new files (not yet in chunks table)
new_files = [f for f in volume_pdfs if f["filename"] not in existing_filenames]
print(f"New files to process: {len(new_files)}")

# Find potentially modified files (filename exists but mod_time is newer)
# We track this by comparing against a watermark or just reprocessing all in a target folder
# For overwritten files in 2026-, let's also detect those
overwrite_candidates = [f for f in volume_pdfs 
                        if f["filename"] in existing_filenames 
                        and "/2026-/" in f["path"]]
print(f"Potential overwrites to reprocess (2026- folder): {len(overwrite_candidates)}")

Already processed: 2797 files
New files to process: 63
Potential overwrites to reprocess (2026- folder): 0


In [ ]:
# --- One-time metadata migration -------------------------------------------
# Add document_id / relative_path to the chunks table and backfill existing
# rows, so the app can open each hit's source PDF directly (no volume scan) and
# so semantic results can too once these columns are added to the vector index.
#
# Safe to re-run: ADD COLUMNS is wrapped so a second run is a no-op, and the
# MERGE only fills rows whose metadata is still NULL.
import os
import hashlib

try:
    spark.sql(f"ALTER TABLE {CHUNKS_TABLE} ADD COLUMNS (document_id STRING, relative_path STRING)")
    print("Added document_id / relative_path columns.")
except Exception as e:
    print(f"Columns likely already present: {e}")

def _rel(path):
    local = path.replace("dbfs:/Volumes/", "/Volumes/").replace("dbfs:", "")
    try:
        return os.path.relpath(local, VOLUME_PATH)
    except Exception:
        return os.path.basename(local)

# Map each known basename to a relative path. Duplicate basenames across folders
# resolve to the first seen (the only ambiguous case) and are counted below.
fname_to_rel = {}
dupes = 0
for f in volume_pdfs:
    if f["filename"] in fname_to_rel:
        dupes += 1
        continue
    fname_to_rel[f["filename"]] = _rel(f["path"])
print(f"Backfill map: {len(fname_to_rel)} filenames ({dupes} duplicate basenames ignored)")

backfill_rows = [
    (fn, rel, "doc_" + hashlib.sha256(rel.encode("utf-8")).hexdigest()[:24])
    for fn, rel in fname_to_rel.items()
]
backfill_df = spark.createDataFrame(
    backfill_rows, "filename string, relative_path string, document_id string"
)
backfill_df.createOrReplaceTempView("_pa_path_backfill")

spark.sql(f"""
    MERGE INTO {CHUNKS_TABLE} t
    USING _pa_path_backfill m ON t.filename = m.filename
    WHEN MATCHED AND (t.relative_path IS NULL OR t.document_id IS NULL)
    THEN UPDATE SET t.relative_path = m.relative_path, t.document_id = m.document_id
""")
print("Backfilled relative_path / document_id for existing rows.")

In [0]:
%pip install pypdf -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from pypdf import PdfReader
import hashlib
import os

def relative_path_for(file_path):
    """Path of the PDF relative to the volume root, e.g. '2009-2025/2018/Name.pdf'.

    This is what lets the app open a search hit's source PDF: the chunks table
    stored only a bare basename before, but PDFs are nested by era/year.
    """
    local_path = file_path.replace("dbfs:/Volumes/", "/Volumes/").replace("dbfs:", "")
    if not local_path.startswith("/Volumes/"):
        local_path = f"/Volumes/{local_path.lstrip('/')}" if "Volumes" not in local_path else local_path
    try:
        return os.path.relpath(local_path, VOLUME_PATH)
    except Exception:
        return os.path.basename(local_path)

def document_id_for(relative_path):
    """Stable document id derived from the volume-relative path (not random).

    Keyed on relative_path (not filename) so duplicate basenames in different
    folders get distinct ids.
    """
    return "doc_" + hashlib.sha256(relative_path.encode("utf-8")).hexdigest()[:24]

def parse_pdf_to_chunks(file_path, filename):
    """Parse a PDF and return text chunks (one per page)."""
    chunks = []
    try:
        # Convert dbfs path to local FUSE path for Volumes
        local_path = file_path.replace("dbfs:/Volumes/", "/Volumes/").replace("dbfs:", "")
        if not local_path.startswith("/Volumes/"):
            local_path = f"/Volumes/{local_path.lstrip('/')}" if "Volumes" not in local_path else local_path

        rel_path = relative_path_for(file_path)
        doc_id = document_id_for(rel_path)

        reader = PdfReader(local_path)

        for page_num, page in enumerate(reader.pages, 1):
            text = page.extract_text()
            if text and text.strip():
                # Generate stable chunk_id
                chunk_id = hashlib.sha256(
                    f"{filename}|{page_num}|0|paragraph|{text[:200]}".encode()
                ).hexdigest()

                chunks.append({
                    "chunk_id": chunk_id,
                    "document_id": doc_id,
                    "filename": filename,
                    "relative_path": rel_path,
                    "page_number": page_num,
                    "chunk_type": "paragraph",
                    "chunk_text": text
                })
    except Exception as e:
        print(f"  ERROR parsing {filename}: {e}")

    return chunks

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), True),
    StructField("filename", StringType(), True),
    StructField("relative_path", StringType(), True),
    StructField("page_number", IntegerType(), True),
    StructField("chunk_type", StringType(), True),
    StructField("chunk_text", StringType(), True),
])

all_new_chunks = []

if new_files:
    print(f"Processing {len(new_files)} new files...")
    for i, f in enumerate(new_files):
        chunks = parse_pdf_to_chunks(f["path"], f["filename"])
        all_new_chunks.extend(chunks)
        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{len(new_files)} files ({len(all_new_chunks)} chunks so far)")

    print(f"\nTotal new chunks: {len(all_new_chunks)}")

    if all_new_chunks:
        new_df = spark.createDataFrame(all_new_chunks, schema=schema)
        # mergeSchema lets the new document_id / relative_path columns land in the
        # existing table the first time this runs.
        new_df.write.mode("append").option("mergeSchema", "true").saveAsTable(CHUNKS_TABLE)
        print(f"Appended {len(all_new_chunks)} chunks to {CHUNKS_TABLE}")
else:
    print("No new files to process.")

In [ ]:
overwrite_chunks = []

if overwrite_candidates:
    print(f"Reprocessing {len(overwrite_candidates)} overwritten files from 2026- folder...")

    # Delete old chunks for these files
    overwrite_filenames = [f["filename"] for f in overwrite_candidates]
    filenames_sql = ",".join([f"'{fn}'" for fn in overwrite_filenames])
    spark.sql(f"DELETE FROM {CHUNKS_TABLE} WHERE filename IN ({filenames_sql})")
    print(f"  Deleted old chunks for {len(overwrite_filenames)} files")

    # Re-parse and insert
    for i, f in enumerate(overwrite_candidates):
        chunks = parse_pdf_to_chunks(f["path"], f["filename"])
        overwrite_chunks.extend(chunks)
        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{len(overwrite_candidates)} files")

    if overwrite_chunks:
        ow_df = spark.createDataFrame(overwrite_chunks, schema=schema)
        ow_df.write.mode("append").option("mergeSchema", "true").saveAsTable(CHUNKS_TABLE)
        print(f"  Re-inserted {len(overwrite_chunks)} chunks")
else:
    print("No overwritten files to reprocess.")

## Exposing `document_id` / `relative_path` in semantic results

Adding the columns to the **source table** (above) is enough for **keyword search**
to open source PDFs. For **semantic** results to do the same, the new columns must
also be **returnable by the vector index**.

`sync_index` (previous cell) refreshes data but does **not** change which columns
the index returns. To expose the new columns, recreate the Delta-Sync index with
them included, e.g.:

```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# One-time: delete and recreate so the new columns are indexed/returnable.
# w.vector_search_indexes.delete_index(index_name=VS_INDEX)
w.vector_search_indexes.create_delta_sync_index(
    endpoint_name="pa-appeals-search-endpoint",
    index_name=VS_INDEX,
    primary_key="chunk_id",
    source_table_name=CHUNKS_TABLE,
    pipeline_type="TRIGGERED",
    embedding_source_columns=[{
        "name": "chunk_text",
        "embedding_model_endpoint_name": "databricks-bge-large-en",  # use your existing model
    }],
    columns_to_sync=[
        "chunk_id", "document_id", "filename", "relative_path",
        "page_number", "chunk_type", "chunk_text",
    ],
)
```

Confirm the embedding model endpoint and `columns_to_sync` support against your
workspace before running — recreating the index re-embeds all rows. Once the
index returns `document_id` / `relative_path`, the app's semantic results can
open the original PDF at the matching page (a small follow-up app change).

In [0]:
from databricks.sdk import WorkspaceClient

total_processed = len(all_new_chunks) + len(overwrite_chunks)

if total_processed > 0:
    w = WorkspaceClient()
    w.vector_search_indexes.sync_index(index_name=VS_INDEX)
    print(f"\nVector index sync triggered for {VS_INDEX}")
    print(f"Summary: {len(new_files)} new files, {len(overwrite_candidates)} overwrites, {total_processed} total chunks processed.")
else:
    print("\nNo changes detected. Index sync skipped.")


Vector index sync triggered for tws_ro_region5.rcd.pa_appeals_chunks_vs_index
Summary: 63 new files, 0 overwrites, 223 total chunks processed.


# Power Automate → Databricks Integration Guide

## Prerequisites
1. A **Databricks Personal Access Token (PAT)** — generate one at: `User Settings > Developer > Access Tokens`
2. **Power Automate Premium** (required for HTTP connector)
3. Access to the SharePoint site/library where PA Appeals PDFs land

---

## Step 1: Create the Power Automate Flow

In Power Automate, create a new **Automated Cloud Flow** with the trigger:

> **"When a file is created (properties only)"** — SharePoint connector

Configure:
- **Site Address**: Your SharePoint site URL (e.g., `https://fema.sharepoint.com/sites/PA-Appeals`)
- **Library Name**: The document library where new appeal PDFs are uploaded
- **Folder**: (Optional) Specific folder path within the library

---

## Step 2: Add a Condition — Filter for PDFs Only

Add a **Condition** action:
- `File name` → **ends with** → `.pdf`

This ensures only PDF files trigger the pipeline.

---

## Step 3: Get File Content from SharePoint

Inside the "If yes" branch, add:

> **"Get file content"** — SharePoint connector

Configure:
- **Site Address**: Same as Step 1
- **File Identifier**: Use the dynamic content `Identifier` from the trigger

---

## Step 4: Upload File to Databricks Volume

Add an **HTTP** action (Premium connector):

| Field | Value |
| --- | --- |
| **Method** | `PUT` |
| **URI** | `https://adb-5672234203219303.3.azuredatabricks.net/api/2.0/fs/files/Volumes/tws_ro_region5/rcd/pa_second_appeals/2026-/@{triggerOutputs()?['body/{FilenameWithExtension}']}` |
| **Headers** | `Authorization`: `Bearer <YOUR_PAT_TOKEN>` |
| | `Content-Type`: `application/octet-stream` |
| **Body** | Dynamic content: **File Content** (from Step 3) |

**Important notes:**
- The filename in the URI must be URL-encoded if it contains spaces or special characters
- Use `encodeUriComponent(triggerOutputs()?['body/{FilenameWithExtension}'])` in the URI expression
- Full URI expression: 
  ```
  concat('https://adb-5672234203219303.3.azuredatabricks.net/api/2.0/fs/files/Volumes/tws_ro_region5/rcd/pa_second_appeals/2026-/', encodeUriComponent(triggerOutputs()?['body/{FilenameWithExtension}']))
  ```

---

## Step 5: Trigger the Processor Notebook (Optional)

Add another **HTTP** action after the upload:

| Field | Value |
| --- | --- |
| **Method** | `POST` |
| **URI** | `https://adb-5672234203219303.3.azuredatabricks.net/api/2.1/jobs/runs/submit` |
| **Headers** | `Authorization`: `Bearer <YOUR_PAT_TOKEN>` |
| | `Content-Type`: `application/json` |
| **Body** | See JSON below |

```json
{
  "run_name": "PA Appeals Auto Refresh",
  "tasks": [{
    "task_key": "process_pdfs",
    "notebook_task": {
      "notebook_path": "/Users/0492734585@fema.dhs.gov/pa-appeals-and-policy-search/PA Appeals PDF Incremental Processor",
      "source": "WORKSPACE"
    },
    "environment_key": "Default"
  }],
  "environments": [{
    "environment_key": "Default",
    "spec": {
      "client": "2",
      "dependencies": ["pypdf"]
    }
  }]
}
```

**Tip:** To avoid triggering the notebook on every single file (if batch uploads happen), add a **Delay** of 5 minutes before this step, or use a **Scheduled Flow** that runs every hour instead of per-file.

---

## Step 6: (Optional) Add Error Handling

After each HTTP action, add a **Condition** to check:
- Upload: `Status code` → **is equal to** → `200`
- Job trigger: `Status code` → **is equal to** → `200`

On failure, send a notification (Teams message, email, etc.).

---

## Alternative: Batch Approach (Recommended)

Instead of triggering per-file, create a **Scheduled Flow** (e.g., daily at 8 AM):

1. **List files** in SharePoint library (filter: created in last 24 hours)
2. **For each** new PDF:
   - Get file content
   - Upload to Databricks Volume (Step 4 above)
3. **After the loop**: Trigger the notebook once (Step 5 above)

This is more efficient and avoids redundant notebook runs.

---

## Security: Storing the PAT Token in Azure Key Vault

**Do NOT hardcode** the PAT in your Power Automate flow. Store it in Azure Key Vault and retrieve it at runtime.

---

### Step A: Generate a Databricks Personal Access Token (PAT)

1. In Databricks, go to **User Settings** (top-right avatar → Settings)
2. Click **Developer** → **Access tokens**
3. Click **Manage** → **Generate new token**
4. Set a description: `Power Automate - PA Appeals Upload`
5. Set a lifetime (e.g., 90 days — set a calendar reminder to rotate)
6. Click **Generate** and **copy the token immediately** (you won’t see it again)

---

### Step B: Request Key Vault Access

If you cannot add secrets to your team’s Key Vault, email `fema-femadex-helpdesk@fema.dhs.gov`:

> **Subject:** Azure Key Vault — Request to Add Databricks Secret for Power Automate Integration
>
> Hi,
>
> I’m building a Power Automate flow that uploads PA Second Appeal PDFs from SharePoint
> to a Databricks UC Volume and triggers a processing job. The flow needs a Databricks PAT
> stored securely in Azure Key Vault for the HTTP connector.
>
> I noticed there’s already a Databricks key in our vault, but I don’t have permission to
> add new secrets. Two questions:
>
> 1. Can I use the existing Databricks key for my flow? It would need permissions to write
>    files to `/Volumes/tws_ro_region5/rcd/pa_second_appeals/2026-/` and submit job runs
>    via the Jobs API.
>
> 2. If not, could a dedicated secret be added for this use case? I can provide a scoped
>    PAT — I’d just need write access to the vault, or someone to add it on my behalf.
>
> Happy to set up a quick call if easier. Thanks!
>
> Caleb Smith — 0492734585@fema.dhs.gov

---

### Step C: Add the Secret to Key Vault (once you have access)

**If you can add the secret yourself:**

1. Go to **portal.azure.com**
2. Search “Key vaults” in the top search bar
3. Open your team’s vault
4. Click **Secrets** (left sidebar) → **+ Generate/Import**
5. Name: `databricks-pat-pa-appeals`
6. Value: paste your Databricks PAT token
7. Content type (optional): `text/plain`
8. Click **Create**

**If an admin adds it for you:**

- Provide them the secret name (`databricks-pat-pa-appeals`)
- Ask them to grant your Power Automate connection (your Azure AD identity) the
  **Secret > Get** permission on the vault’s Access Policies
- They do NOT need your actual token value until you securely provide it (e.g., via
  Teams chat with disappearing messages, or have them watch you paste it in a screen share)

---

### Step D: Connect Key Vault to Power Automate

1. In your Power Automate flow, add the action: **Azure Key Vault → Get secret**
2. First time: it will ask you to create a connection — sign in with your FEMA Azure AD account
3. Select your vault name and secret name (`databricks-pat-pa-appeals`)
4. In your HTTP actions (upload file, trigger job), set the Authorization header to:
   ```
   Bearer @{body('Get_secret')?['value']}
   ```
   (Replace `'Get_secret'` with whatever Power Automate names that action, e.g., `'Get_secret_2'`)

---

### Step E: Secure the Flow

1. On each HTTP action: click `...` → **Settings** → enable **Secure Inputs** and **Secure Outputs**
   - This prevents the token from appearing in flow run history
2. On the Key Vault action: enable **Secure Outputs**
3. Limit who can edit the flow: **Flow details** → **Owners** — only add people who should see the config

---

### Step F: Token Rotation

- PATs expire after the lifetime you set. Put a reminder on your calendar.
- To rotate: generate a new PAT in Databricks, update the secret value in Key Vault
  (Secrets → select secret → New Version → paste new token)
- The flow picks up the new value automatically on next run — no flow edits needed.

---

## Testing

1. Upload a test PDF to your SharePoint library
2. Watch the flow run in Power Automate → Run History
3. Verify the file appears in the volume: `/Volumes/tws_ro_region5/rcd/pa_second_appeals/2026-/`
4. Check the notebook run completed: Databricks → Workflows → Job Runs
5. Search for the new document in the app

---

## Troubleshooting

| Issue | Fix |
| --- | --- |
| Key Vault “Get secret” fails with 403 | Your Azure AD identity needs **Secret > Get** permission on the vault’s Access Policies. Ask your vault admin. |
| HTTP upload returns 401 Unauthorized | PAT expired or was revoked. Rotate it (Step F). |
| HTTP upload returns 403 Forbidden | PAT user lacks `WRITE VOLUME` permission on `tws_ro_region5.rcd.pa_second_appeals`. |
| Job trigger returns 403 | PAT user lacks permission to run notebooks or submit jobs. |
| Flow runs but PDF not searchable | Check the processor notebook ran and the vector index synced (use Admin tab in app). |
| Key Vault not visible in Power Automate | Your Azure AD account may not have Reader role on the vault’s resource group. Ask your admin. |